In [1]:
import pandas as pd
import numpy as np
import heapq
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_validate
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 1. EXPANDING MEDIAN CLASS (WITH TWO HEAPS)
# ------------------------------------------------------------

class ExpandingMedian:
    """Class for calculating expanding median using two heaps"""
    def __init__(self):
        self.min_heap = []  # right half (larger values)
        self.max_heap = []  # left half (smaller values, stored as negatives)
        self.n = 0
        
    def add(self, x):
        """Add a new value and update heaps"""
        self.n += 1
        
        # Add to appropriate heap
        if not self.max_heap or x <= -self.max_heap[0]:
            heapq.heappush(self.max_heap, -x)
        else:
            heapq.heappush(self.min_heap, x)
        
        # Balance heaps
        if len(self.max_heap) > len(self.min_heap) + 1:
            heapq.heappush(self.min_heap, -heapq.heappop(self.max_heap))
        elif len(self.min_heap) > len(self.max_heap):
            heapq.heappush(self.max_heap, -heapq.heappop(self.min_heap))
    
    def get_median(self):
        """Get current median"""
        if self.n == 0:
            return 0.0
        if self.n % 2 == 1:
            return -self.max_heap[0]
        else:
            return (-self.max_heap[0] + self.min_heap[0]) / 2.0

# ------------------------------------------------------------
# 2. TREND SCORE CALCULATION FUNCTIONS
# ------------------------------------------------------------

def normalized_global_slope(window):
    """Calculate normalized global slope between -1 and 1"""
    if len(window) < 2:
        return 0
    x = np.arange(len(window))
    slope = np.polyfit(x, window, 1)[0]
    max_possible_slope = (np.max(window) - np.min(window)) / len(window) if len(window) > 1 else 1
    if max_possible_slope == 0:
        return 0
    return slope / max_possible_slope

def pairwise_slopes(window):
    """Calculate slopes between all point pairs"""
    n = len(window)
    slopes = []
    for i in range(n):
        for j in range(i+1, n):
            if j > i:
                slope = (window[j] - window[i]) / (j - i) if (j - i) != 0 else 0
                slopes.append(slope)
    return np.array(slopes)

def consistency_factor(P):
    """Consistency factor based on slope variance"""
    if len(P) == 0:
        return 0
    abs_slopes = np.abs(P)
    if np.mean(abs_slopes) == 0:
        return 0
    cv = np.std(abs_slopes) / np.mean(abs_slopes)
    return 1 / (1 + cv)

def contradiction_factor(G, P):
    """Contradiction factor between global and pairwise slopes"""
    if len(P) == 0:
        return 0
    if G >= 0:
        contradictions = np.sum(P < 0)
    else:
        contradictions = np.sum(P > 0)
    return contradictions / len(P)

def calculate_trend_score(window):
    """Calculate final trend score"""
    if len(window) < 2:  # Minimum 2 points needed
        return 0.0
        
    G = normalized_global_slope(window)
    P = pairwise_slopes(window)
    CF = consistency_factor(P)
    CD = contradiction_factor(G, P)
    
    G = np.clip(G, -1, 1)
    CF = np.clip(CF, 0, 1)
    CD = np.clip(CD, 0, 1)
    
    Trend_Score = G * CF * (1 - CD)
    return np.clip(Trend_Score, -1, 1)

# ------------------------------------------------------------
# 3. FIXED: TREND SCORE COMPUTATION WITH PROPER WINDOW HANDLING
# ------------------------------------------------------------

def compute_incremental_trend_scores(df, feature_columns, window_size=10):
    """
    FIXED: Proper window_size handling
    For first window_size-1 samples, use all available data
    """
    n_samples = len(df)
    n_features = len(feature_columns)
    
    # Initialize output matrix
    trend_scores = np.zeros((n_samples, n_features))
    
    for f_idx, feature in enumerate(feature_columns):
        data = df[feature].values
        
        for i in range(n_samples):
            if i < window_size - 1:
                # Not enough data for full window - use all available data
                window = data[:i+1]
                if len(window) >= 2:  # Need at least 2 points
                    score = calculate_trend_score(window)
                    trend_scores[i, f_idx] = score
            else:
                # Have enough data - use exactly window_size samples
                window = data[i-window_size+1:i+1]
                score = calculate_trend_score(window)
                trend_scores[i, f_idx] = score
    
    # Create column names
    columns = [f'{feature}_trend' for feature in feature_columns]
    
    return pd.DataFrame(trend_scores, columns=columns, index=df.index)

# ------------------------------------------------------------
# 4. FIXED: FEATURE SELECTION WITH PROPER WINDOW HANDLING
# ------------------------------------------------------------

def select_top_trend_features(df, feature_columns, window_size=10, top_n=3):
    """
    FIXED: Calculate trend scores and select top features
    Only consider samples with full windows for strength calculation
    """
    print(f"\n[FEATURE SELECTION] Analyzing all {len(feature_columns)} features...")
    print(f"  Window size: {window_size}")
    
    # Calculate trend scores
    trend_scores_df = compute_incremental_trend_scores(df, feature_columns, window_size=window_size)
    
    # Calculate trend strength for each feature
    trend_strength = {}
    for feature in feature_columns:
        trend_col = f'{feature}_trend'
        if trend_col in trend_scores_df.columns:
            # Only use samples from window_size-1 onward (where we have full windows)
            if len(trend_scores_df) >= window_size:
                valid_scores = trend_scores_df[trend_col].iloc[window_size-1:]
                strength = np.abs(valid_scores).mean()
            else:
                strength = np.abs(trend_scores_df[trend_col]).mean()
            trend_strength[feature] = strength
    
    # Sort descending and select top_n features
    sorted_features = sorted(trend_strength.items(), key=lambda x: x[1], reverse=True)
    top_features = [f[0] for f in sorted_features[:top_n]]
    
    print(f"\n   Trend strength scores (higher = stronger trend):")
    for feature, strength in sorted_features:
        marker = "✓ SELECTED" if feature in top_features else "  "
        print(f"   {marker} {feature}: {strength:.4f}")
    
    print(f"\n   Selected top {top_n} features for clustering: {top_features}")
    
    # Return trend scores only for selected features
    selected_trend_cols = [f'{f}_trend' for f in top_features]
    
    return top_features, trend_scores_df[selected_trend_cols]

# ------------------------------------------------------------
# 5. INCREMENTAL STATISTICS FUNCTIONS FOR TREND SCORES
# ------------------------------------------------------------

def compute_incremental_stats_for_trend_scores(df_trend_scores):
    """
    Compute incremental statistics (mean, median, std) for trend scores
    States start fresh for each dataset
    """
    n_samples = len(df_trend_scores)
    n_features = len(df_trend_scores.columns)
    
    # Initialize output matrix
    incremental_stats = np.zeros((n_samples, n_features * 3))
    
    # Initialize fresh states for each trend feature
    states = []
    for _ in range(n_features):
        states.append({
            'n': 0,
            'mean': 0.0,
            'M2': 0.0,  # For variance calculation (Welford's algorithm)
            'median_calculator': ExpandingMedian()
        })
    
    # Process each sample sequentially
    for i in range(n_samples):
        for f_idx, feature in enumerate(df_trend_scores.columns):
            # Get current trend score value
            x = df_trend_scores.iloc[i][feature]
            state = states[f_idx]
            
            # Update count
            state['n'] += 1
            n = state['n']
            
            # Update expanding mean (recursive formula)
            delta = x - state['mean']
            state['mean'] += delta / n
            
            # Update expanding variance (Welford's algorithm)
            delta2 = x - state['mean']
            state['M2'] += delta * delta2
            
            # Update expanding median
            state['median_calculator'].add(x)
            median_val = state['median_calculator'].get_median()
            
            # Calculate expanding std
            if n > 1:
                variance = state['M2'] / (n - 1)
                std_val = np.sqrt(variance)
            else:
                std_val = 0.0
            
            # Store results
            col_idx = f_idx * 3
            incremental_stats[i, col_idx] = state['mean']
            incremental_stats[i, col_idx + 1] = median_val
            incremental_stats[i, col_idx + 2] = std_val
    
    # Create column names
    columns = []
    for feature in df_trend_scores.columns:
        base_name = feature.replace('_trend', '')
        columns.extend([f'{base_name}_trend_mean', f'{base_name}_trend_median', f'{base_name}_trend_std'])
    
    return pd.DataFrame(incremental_stats, columns=columns, index=df_trend_scores.index)

# ------------------------------------------------------------
# 6. UNIVERSAL EDGE-PRIORITY BALANCED CLUSTERING
# ------------------------------------------------------------

def ensure_label_coverage_in_clusters(clusters, labels, X_features=None, min_threshold=10, target_count=1000):
    """
    FORCE each cluster to have at least target_count of each label
    Copy samples from clusters that have enough samples
    """
    np.random.seed(42)
    clusters = np.array(clusters)
    labels = np.array(labels)
    
    n_clusters = len(np.unique(clusters))
    unique_labels = np.unique(labels)
    
    # Create a copy of clusters to modify
    modified_clusters = clusters.copy()
    
    print(f"\n" + "="*60)
    print("FORCED LABEL COVERAGE ENHANCEMENT")
    print("="*60)
    print(f"Target: Each cluster must have at least {target_count} samples of each label")
    
    # Count initial distribution
    initial_dist = np.zeros((n_clusters, len(unique_labels)), dtype=int)
    for c in range(n_clusters):
        for l_idx, l_val in enumerate(unique_labels):
            initial_dist[c, l_idx] = np.sum((modified_clusters == c) & (labels == l_val))
    
    print("\nINITIAL DISTRIBUTION:")
    for c in range(n_clusters):
        print(f"  Cluster {c}:")
        for l_idx, l_val in enumerate(unique_labels):
            print(f"    {l_val}: {initial_dist[c, l_idx]}")
    
    # Track copied samples
    copied_info = {
        'total_copied': 0,
        'details': []
    }
    
    # Process each target cluster
    for target_cluster in range(n_clusters):
        print(f"\n" + "-"*40)
        print(f"Processing Cluster {target_cluster}")
        print("-"*40)
        
        # Process each label
        for label_idx, label_val in enumerate(unique_labels):
            current_count = np.sum((modified_clusters == target_cluster) & (labels == label_val))
            
            if current_count < target_count:
                needed = target_count - current_count
                print(f"\n  Label {label_val}: has {current_count}, needs {needed}")
                
                # Find source clusters with this label
                source_candidates = []
                for source_cluster in range(n_clusters):
                    if source_cluster != target_cluster:
                        source_count = np.sum((modified_clusters == source_cluster) & (labels == label_val))
                        if source_count > needed:  # Source has enough to share
                            source_candidates.append((source_cluster, source_count))
                
                if not source_candidates:
                    # If no cluster has enough, take from any cluster that has this label
                    for source_cluster in range(n_clusters):
                        if source_cluster != target_cluster:
                            source_count = np.sum((modified_clusters == source_cluster) & (labels == label_val))
                            if source_count > 0:
                                source_candidates.append((source_cluster, source_count))
                
                if source_candidates:
                    # Sort by distance if X_features provided
                    if X_features is not None:
                        # Calculate cluster centers
                        centers = {}
                        for c in range(n_clusters):
                            points = X_features[modified_clusters == c]
                            if len(points) > 0:
                                centers[c] = np.mean(points, axis=0)
                        
                        # Sort by distance to target cluster
                        if target_cluster in centers:
                            target_center = centers[target_cluster]
                            source_candidates.sort(key=lambda x: np.linalg.norm(target_center - centers.get(x[0], target_center)))
                    
                    # Take samples from source clusters
                    total_taken = 0
                    for source_cluster, source_count in source_candidates:
                        if total_taken >= needed:
                            break
                        
                        # How many to take from this source
                        still_needed = needed - total_taken
                        take_from_this = min(still_needed, source_count // 2)  # Take max half of source's samples
                        
                        if take_from_this > 0:
                            # Find indices of this label in source cluster
                            source_indices = np.where((modified_clusters == source_cluster) & (labels == label_val))[0]
                            
                            # Select samples to copy
                            n_to_take = min(take_from_this, len(source_indices))
                            selected = np.random.choice(source_indices, n_to_take, replace=False)
                            
                            # Copy them to target cluster
                            modified_clusters[selected] = target_cluster
                            
                            total_taken += n_to_take
                            copied_info['total_copied'] += n_to_take
                            copied_info['details'].append({
                                'from': source_cluster,
                                'to': target_cluster,
                                'label': label_val,
                                'count': n_to_take
                            })
                            
                            print(f"    Copied {n_to_take} from cluster {source_cluster}")
                    
                    if total_taken > 0:
                        print(f"    Total copied for {label_val}: {total_taken}")
                    else:
                        print(f"    WARNING: Could not copy any samples for {label_val}")
                else:
                    print(f"    ERROR: No source cluster has label {label_val}")
    
    # Final distribution
    final_dist = np.zeros((n_clusters, len(unique_labels)), dtype=int)
    for c in range(n_clusters):
        for l_idx, l_val in enumerate(unique_labels):
            final_dist[c, l_idx] = np.sum((modified_clusters == c) & (labels == l_val))
    
    print("\n" + "="*60)
    print("FINAL DISTRIBUTION")
    print("="*60)
    for c in range(n_clusters):
        print(f"\nCluster {c}:")
        for l_idx, l_val in enumerate(unique_labels):
            status = "✅" if final_dist[c, l_idx] >= target_count else f"⚠ ({final_dist[c, l_idx]}/{target_count})"
            print(f"  {l_val}: {final_dist[c, l_idx]} {status}")
    
    print(f"\nTotal samples copied: {copied_info['total_copied']}")
    
    return modified_clusters, copied_info

def balanced_cluster_assignment_with_edge_priority(X, y, kmeans_model, 
                                                   edge_threshold=0.15, 
                                                   alpha=0.3, beta=0.7,
                                                   top_k_clusters=3,
                                                   ensure_label_coverage=True,
                                                   min_threshold=10,
                                                   target_count=1000):
    """
    Universal cluster assignment for any number of labels
    Used only during training
    """
    n_samples = X.shape[0]
    n_clusters = kmeans_model.n_clusters
    
    distances = kmeans_model.transform(X)
    initial_clusters = np.argmin(distances, axis=1)
    
    unique_labels = np.unique(y)
    n_labels = len(unique_labels)
    label_distribution = np.zeros((n_clusters, n_labels))
    
    for cluster_id in range(n_clusters):
        cluster_indices = np.where(initial_clusters == cluster_id)[0]
        if len(cluster_indices) > 0:
            cluster_labels = y[cluster_indices]
            for label_idx, label_val in enumerate(unique_labels):
                label_distribution[cluster_id, label_idx] = np.sum(cluster_labels == label_val)
    
    final_clusters = np.copy(initial_clusters)
    
    for i in range(n_samples):
        sample_distances = distances[i]
        sample_label = y[i]
        label_idx = np.where(unique_labels == sample_label)[0][0]
        
        sorted_indices = np.argsort(sample_distances)
        
        if len(sorted_indices) >= 2:
            closest_dist = sample_distances[sorted_indices[0]]
            second_dist = sample_distances[sorted_indices[1]]
            
            if closest_dist > 0:
                diff_ratio = (second_dist - closest_dist) / closest_dist
            else:
                diff_ratio = 1.0
            
            if diff_ratio < edge_threshold:
                K = min(top_k_clusters, n_clusters)
                candidate_clusters = sorted_indices[:K]
                
                scores = []
                for cluster_id in candidate_clusters:
                    norm_distance = 1 - (sample_distances[cluster_id] / 
                                       np.max(sample_distances[candidate_clusters]))
                    
                    max_label_count = np.max(label_distribution[:, label_idx])
                    if max_label_count > 0:
                        label_needs = 1 - (label_distribution[cluster_id, label_idx] / max_label_count)
                    else:
                        label_needs = 1
                    
                    score = alpha * norm_distance + beta * label_needs
                    scores.append(score)
                
                best_idx = np.argmax(scores)
                final_clusters[i] = candidate_clusters[best_idx]
                
                label_distribution[final_clusters[i], label_idx] += 1
    
    # Ensure label coverage in all clusters
    if ensure_label_coverage and n_labels > 1:
        print("\n" + "="*60)
        print("ENFORCING LABEL COVERAGE IN CLUSTERS")
        print("="*60)
        final_clusters, copy_info = ensure_label_coverage_in_clusters(
            final_clusters, y, X, 
            min_threshold=min_threshold,
            target_count=target_count
        )
    
    return final_clusters

def analyze_cluster_balance(df_with_clusters, cluster_col='cluster', label_col='label'):
    """Universal cluster balance analysis"""
    n_clusters = df_with_clusters[cluster_col].nunique()
    unique_labels = sorted(df_with_clusters[label_col].unique())
    n_labels = len(unique_labels)
    
    print("="*60)
    print("CLUSTER BALANCE ANALYSIS")
    print("="*60)
    print(f"Number of clusters: {n_clusters}")
    print(f"Number of unique labels: {n_labels}")
    print(f"Labels: {unique_labels}")
    
    distribution_matrix = pd.DataFrame(
        index=[f'Cluster {i}' for i in range(n_clusters)],
        columns=[f'Label {str(l)}' for l in unique_labels]
    )
    
    balance_scores = []
    cluster_info = {}
    
    for cluster_id in range(n_clusters):
        cluster_data = df_with_clusters[df_with_clusters[cluster_col] == cluster_id]
        total_in_cluster = len(cluster_data)
        
        if total_in_cluster > 0:
            row_data = []
            for label in unique_labels:
                count = len(cluster_data[cluster_data[label_col] == label])
                percentage = (count / total_in_cluster) * 100
                distribution_matrix.loc[f'Cluster {cluster_id}', f'Label {str(label)}'] = f"{count} ({percentage:.1f}%)"
                row_data.append(count)
            
            proportions = np.array(row_data) / total_in_cluster
            
            if n_labels > 1:
                gini_impurity = 1 - np.sum(proportions**2)
                normalized_gini = gini_impurity / (1 - 1/n_labels)
            else:
                normalized_gini = 0
            
            balance_scores.append(normalized_gini)
            
            cluster_info[cluster_id] = {
                'total_samples': total_in_cluster,
                'balance_score': normalized_gini,
                'label_distribution': dict(zip(unique_labels, row_data))
            }
            
            print(f"\nCluster {cluster_id}:")
            print(f"  Total samples: {total_in_cluster}")
            if n_labels > 1:
                print(f"  Balance score: {normalized_gini:.3f} (1.0 = perfectly balanced)")
            for label, count, prop in zip(unique_labels, row_data, proportions):
                print(f"  Label {label}: {count} samples ({prop*100:.1f}%)")
    
    print("\n" + "="*60)
    print("DISTRIBUTION MATRIX")
    print("="*60)
    print(distribution_matrix)
    
    if balance_scores:
        avg_balance = np.mean(balance_scores)
        print(f"\nAverage balance score: {avg_balance:.3f}")
    else:
        avg_balance = 0
    
    return distribution_matrix, avg_balance, cluster_info

# ------------------------------------------------------------
# 7. FIXED: MAIN PIPELINE WITH PROPER WINDOW SIZE HANDLING
# ------------------------------------------------------------

def universal_trend_clustering_pipeline(df, 
                                       target_column='label',
                                       n_clusters=3,
                                       top_n_features=3,
                                       window_size=10,
                                       test_size=0.2,
                                       random_state=42,
                                       external_test_df=None,
                                       preserve_order=True,
                                       ensure_label_coverage=True,
                                       min_threshold=10,
                                       target_count=1):
    """
    FIXED: Universal pipeline with proper window_size handling
    window_size is now passed consistently throughout the pipeline
    """
    
    print("="*80)
    print("UNIVERSAL TREND CLUSTERING & CLASSIFICATION PIPELINE")
    print("="*80)
    print(f"USING TOP {top_n_features} FEATURES WITH HIGHEST TREND STRENGTH")
    print(f"Each selected feature → 3 statistical features (mean, median, std)")
    print(f"Total clustering features: {top_n_features} × 3 = {top_n_features * 3}")
    print(f"Trend window size: {window_size} (used consistently throughout)")
    print(f"NOTE: Data order preservation = {preserve_order}")
    print(f"NOTE: Label coverage enforcement = {ensure_label_coverage}")
    if ensure_label_coverage:
        print(f"      Enhancement threshold: <{min_threshold} samples")
        print(f"      Target count per label: {target_count} samples")
    
    # --------------------------------------------------------
    # 1. Data Preparation
    # --------------------------------------------------------
    
    df_processed = df.copy()
    
    if target_column not in df_processed.columns:
        available_cols = df_processed.columns.tolist()
        print(f"Warning: Target column '{target_column}' not found.")
        print(f"Available columns: {available_cols}")
        
        if 'label' in available_cols:
            target_column = 'label'
            print(f"Using 'label' as target column instead.")
        else:
            raise ValueError(f"Target column not found. Available: {available_cols}")
    
    all_features = [col for col in df_processed.columns if col != target_column]
    
    if not all_features:
        raise ValueError("No feature columns found!")
    
    print(f"\n[1] DATA PREPARATION")
    print(f"    Target column: {target_column}")
    print(f"    Number of features: {len(all_features)}")
    print(f"    Total samples: {len(df_processed)}")
    
    label_info = df_processed[target_column].value_counts().sort_index()
    n_classes = len(label_info)
    print(f"    Number of classes: {n_classes}")
    print(f"    Class distribution:")
    for label, count in label_info.items():
        percentage = (count / len(df_processed)) * 100
        print(f"      Class {label}: {count} samples ({percentage:.1f}%)")
    
    # --------------------------------------------------------
    # 2. Train-Test Split (or use external test)
    # --------------------------------------------------------
    
    print(f"\n[2] DATA SPLITTING")
    
    if external_test_df is not None:
        # Use external test set
        print(f"    Using external test set provided")
        train_df = df_processed.copy()
        test_df = external_test_df.copy()
        
        # Ensure test set has same columns as train set
        missing_cols = set(train_df.columns) - set(test_df.columns)
        if missing_cols:
            print(f"    Warning: Test set missing columns: {missing_cols}")
            for col in missing_cols:
                if col != target_column:
                    test_df[col] = 0
        
        # Keep only common columns
        common_cols = list(set(train_df.columns) & set(test_df.columns))
        train_df = train_df[common_cols]
        test_df = test_df[common_cols]
        
        print(f"    Training samples: {len(train_df)}")
        print(f"    Test samples: {len(test_df)}")
        
    else:
        # Internal train-test split
        if preserve_order:
            # Class-wise sequential split (preserves order AND label distribution)
            print(f"    Using CLASS-WISE sequential split (order preserved)")
            
            train_dfs = []
            test_dfs = []
            
            # Get unique labels
            unique_labels = df_processed[target_column].unique()
            print(f"    Splitting each of {len(unique_labels)} classes separately...")
            
            for label in unique_labels:
                # Get data for this class only
                class_data = df_processed[df_processed[target_column] == label].copy()
                n_class_samples = len(class_data)
                
                if n_class_samples > 0:
                    # Calculate split index for this class
                    split_idx = int(n_class_samples * (1 - test_size))
                    
                    # Split this class's data
                    class_train = class_data.iloc[:split_idx]
                    class_test = class_data.iloc[split_idx:]
                    
                    train_dfs.append(class_train)
                    test_dfs.append(class_test)
                    
                    print(f"      Class {label}: {n_class_samples} total -> "
                          f"{len(class_train)} train, {len(class_test)} test")
            
            # Combine all classes
            train_df = pd.concat(train_dfs, ignore_index=False).sort_index()
            test_df = pd.concat(test_dfs, ignore_index=False).sort_index()
            
            print(f"\n    Training samples: {len(train_df)}")
            print(f"    Test samples: {len(test_df)}")
            
            # Display class distribution
            print(f"\n    Class distribution in training set:")
            train_label_counts = train_df[target_column].value_counts().sort_index()
            for label, count in train_label_counts.items():
                percentage = (count / len(train_df)) * 100
                print(f"      Class {label}: {count} samples ({percentage:.1f}%)")
            
            print(f"\n    Class distribution in test set:")
            test_label_counts = test_df[target_column].value_counts().sort_index()
            for label, count in test_label_counts.items():
                percentage = (count / len(test_df)) * 100
                print(f"      Class {label}: {count} samples ({percentage:.1f}%)")
            
        else:
            # Original random split with stratification
            print(f"    Using random split with stratification")
            from sklearn.model_selection import train_test_split
            if n_classes > 1:
                train_df, test_df = train_test_split(
                    df_processed, 
                    test_size=test_size, 
                    stratify=df_processed[target_column], 
                    random_state=random_state
                )
            else:
                print("    Warning: Only one class found. Stratification disabled.")
                train_df, test_df = train_test_split(
                    df_processed, 
                    test_size=test_size, 
                    random_state=random_state
                )
            print(f"    Training samples: {len(train_df)}")
            print(f"    Test samples: {len(test_df)}")
    
    # --------------------------------------------------------
    # 3. FIXED: FEATURE SELECTION WITH PROPER WINDOW SIZE
    # --------------------------------------------------------
    
    print(f"\n[3] FEATURE SELECTION BASED ON TREND STRENGTH")
    print(f"    Analyzing all {len(all_features)} features to find top {top_n_features}...")
    print(f"    Using window_size={window_size} for trend calculation")
    
    # Calculate trend scores for training data and select top features
    top_features, top_trend_scores_train = select_top_trend_features(
        train_df, all_features, window_size=window_size, top_n=top_n_features
    )
    
    print(f"\n    Selected top {len(top_features)} features for clustering:")
    for i, feat in enumerate(top_features):
        print(f"      {i+1}. {feat}")
    
    print(f"    These {top_n_features} features will be used for all subsequent calculations")
    print(f"    (trend scores + 3 statistical features each = {top_n_features * 3} total features)")
    
    # --------------------------------------------------------
    # 4. FIXED: Compute INCREMENTAL TREND SCORES with proper window_size
    # --------------------------------------------------------
    
    print(f"\n[4] COMPUTING INCREMENTAL TREND SCORES FOR TOP FEATURES")
    print(f"    Computing trend scores for training data (window_size={window_size})...")
    
    # Compute trend scores for training data (only for top features)
    trend_scores_train = compute_incremental_trend_scores(train_df, top_features, window_size=window_size)
    
    # Compute incremental statistics on trend scores
    incremental_stats_train = compute_incremental_stats_for_trend_scores(trend_scores_train)
    
    train_df_with_stats = pd.concat([train_df, incremental_stats_train], axis=1)
    incremental_feature_cols = [col for col in train_df_with_stats.columns 
                               if col.endswith(('_trend_mean', '_trend_median', '_trend_std'))]
    
    print(f"    Created {len(incremental_feature_cols)} incremental trend features")
    print(f"    (3 statistics × {len(top_features)} features = {len(incremental_feature_cols)} features)")
    
    # --------------------------------------------------------
    # 5. Edge-Priority Balanced Clustering (using only top features' statistics)
    # --------------------------------------------------------
    
    print(f"\n[5] EDGE-PRIORITY BALANCED CLUSTERING")
    print(f"    Using TREND SCORE STATISTICS from top {len(top_features)} features")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_df_with_stats[incremental_feature_cols])
    
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(X_train_scaled)
    
    BALANCING_PARAMS = {
        'edge_threshold': 0.15,
        'alpha': 0.3,
        'beta': 0.7,
        'top_k_clusters': 3,
        'ensure_label_coverage': ensure_label_coverage,
        'min_threshold': min_threshold,
        'target_count': target_count
    }
    
    train_clusters_balanced = balanced_cluster_assignment_with_edge_priority(
        X_train_scaled, 
        train_df[target_column].values, 
        kmeans_model=kmeans,
        **BALANCING_PARAMS
    )
    
    train_df_with_stats['cluster'] = train_clusters_balanced
    
    print("\n    Training Cluster Balance Results:")
    balance_matrix, avg_balance, cluster_info = analyze_cluster_balance(
        train_df_with_stats, 'cluster', target_column
    )
    
    train_df_final = train_df_with_stats.drop(columns=incremental_feature_cols)
    
    # --------------------------------------------------------
    # 6. FIXED: Process Test Data with SAME window_size
    # --------------------------------------------------------
    
    print(f"\n[6] PROCESSING TEST DATA")
    print(f"    Using the SAME {len(top_features)} top features selected from training data")
    print(f"    Computing incremental trend scores for test data with window_size={window_size} (SAME as training)")
    
    # Compute trend scores for test data (only for top features)
    trend_scores_test = compute_incremental_trend_scores(test_df, top_features, window_size=window_size)
    
    # Compute incremental statistics on trend scores
    incremental_stats_test = compute_incremental_stats_for_trend_scores(trend_scores_test)
    
    test_df_with_stats = pd.concat([test_df, incremental_stats_test], axis=1)
    X_test_scaled = scaler.transform(test_df_with_stats[incremental_feature_cols])
    
    # Use simple kmeans prediction for test data (no edge priority)
    test_clusters = kmeans.predict(X_test_scaled)
    
    test_df_with_stats['cluster'] = test_clusters
    test_df_final = test_df_with_stats.drop(columns=incremental_feature_cols)
    
    # --------------------------------------------------------
    # 7. Train and Evaluate Models for Each Cluster
    # --------------------------------------------------------
    
    print(f"\n[7] MODEL TRAINING & EVALUATION")
    print(f"    Using ORIGINAL FEATURES for classification (not trend scores)")
    print(f"    Note: Using ALL {len(all_features)} original features for classification")
    
    classifiers = {}
    train_results = {}
    test_results = {}
    feature_importances = {}
    
    for i in range(n_clusters):
        train_cluster_data = train_df_final[train_df_final['cluster'] == i]
        test_cluster_data = test_df_final[test_df_final['cluster'] == i]
        
        if len(train_cluster_data) == 0:
            print(f"\n    Cluster {i}: No training data. Skipping.")
            continue
        
        print(f"\n    Cluster {i}:")
        print(f"      Training samples: {len(train_cluster_data)}")
        if len(test_cluster_data) > 0:
            print(f"      Test samples: {len(test_cluster_data)}")
        else:
            print(f"      Test samples: 0")
        
        # Check label coverage in this cluster
        unique_labels_in_cluster = train_cluster_data[target_column].unique()
        label_counts = train_cluster_data[target_column].value_counts()
        
        print(f"      Label distribution in cluster:")
        for label, count in label_counts.items():
            print(f"        Label {label}: {count} samples")
        
        # Check if any label has less than min_threshold
        labels_below_threshold = [label for label, count in label_counts.items() 
                                 if count < min_threshold]
        if labels_below_threshold:
            print(f"      ⚠ Warning: Labels below threshold ({min_threshold}): {labels_below_threshold}")
        
        # Use ALL original features for classification (excluding target and cluster)
        cluster_features = [col for col in train_cluster_data.columns 
                          if col not in [target_column, 'cluster']]
        
        X_train_cluster = train_cluster_data[cluster_features]
        y_train_cluster = train_cluster_data[target_column]
        
        if len(test_cluster_data) > 0:
            X_test_cluster = test_cluster_data[cluster_features]
            y_test_cluster = test_cluster_data[target_column]
        
        if len(X_train_cluster) < 5:
            print(f"      Warning: Not enough data for cross-validation. Skipping.")
            continue
        
        clf = RandomForestClassifier(
            n_estimators=80,
            random_state=random_state,
            class_weight='balanced' if n_classes > 1 else None
        )
        
        # Cross-validation on training data
        if n_classes > 1:
            cv_results = cross_validate(clf, X_train_cluster, y_train_cluster, 
                                       cv=min(5, len(X_train_cluster)), 
                                       scoring=['accuracy', 'f1_macro', 'recall_macro', 'precision_macro'], 
                                       return_train_score=False)
        
        clf.fit(X_train_cluster, y_train_cluster)
        
        # Store classifier
        classifiers[i] = clf
        
        # Store feature importances
        feature_importances[i] = pd.DataFrame({
            'feature': X_train_cluster.columns,
            'importance': clf.feature_importances_
        }).sort_values('importance', ascending=False)
        
        # Training results
        if n_classes > 1:
            train_results[i] = {
                'cv_accuracy': np.mean(cv_results['test_accuracy']),
                'cv_f1_macro': np.mean(cv_results['test_f1_macro']),
                'cv_recall_macro': np.mean(cv_results['test_recall_macro']),
                'cv_precision_macro': np.mean(cv_results['test_precision_macro']),
                'n_samples': len(train_cluster_data),
                'n_labels': len(unique_labels_in_cluster),
                'label_distribution': label_counts.to_dict()
            }
            
            print(f"      CV Accuracy: {train_results[i]['cv_accuracy']:.4f}")
            print(f"      CV F1 Macro: {train_results[i]['cv_f1_macro']:.4f}")
            print(f"      Labels covered: {train_results[i]['n_labels']}/{n_classes}")
        
        # Test results if test data exists
        if len(test_cluster_data) > 0 and n_classes > 1:
            y_test_pred = clf.predict(X_test_cluster)
            
            test_results[i] = {
                'accuracy': accuracy_score(y_test_cluster, y_test_pred),
                'f1_weighted': f1_score(y_test_cluster, y_test_pred, average='weighted'),
                'recall_weighted': recall_score(y_test_cluster, y_test_pred, average='weighted'),
                'precision_weighted': precision_score(y_test_cluster, y_test_pred, average='weighted'),
                'confusion_matrix': confusion_matrix(y_test_cluster, y_test_pred),
                'n_samples': len(test_cluster_data),
                'class_report': classification_report(y_test_cluster, y_test_pred, digits=3)
            }
    
    # --------------------------------------------------------
    # 8. Display Final Results
    # --------------------------------------------------------
    
    print("\n" + "="*80)
    print("FINAL RESULTS SUMMARY")
    print("="*80)
    
    print(f"\nDataset Information:")
    print(f"  Total samples: {len(df_processed)}")
    print(f"  Number of classes: {n_classes}")
    print(f"  Target column: '{target_column}'")
    print(f"  Top features selected for clustering: {len(top_features)}")
    print(f"    {top_features}")
    print(f"  Features used for clustering (3 stats each): {len(incremental_feature_cols)}")
    print(f"  All original features for classification: {len(all_features)}")
    print(f"  Trend window size: {window_size} (used consistently)")
    print(f"  Label coverage enforcement: {ensure_label_coverage}")
    if ensure_label_coverage:
        print(f"    Enhancement threshold: <{min_threshold} samples")
        print(f"    Target count per label: {target_count} samples")
    
    print(f"\nClustering Results (based on TREND SCORES from top features):")
    print(f"  Number of clusters: {n_clusters}")
    print(f"  Average balance score: {avg_balance:.3f}")
    print(f"  Training cluster distribution:")
    print(train_df_final['cluster'].value_counts().sort_index())
    print(f"  Test cluster distribution:")
    print(test_df_final['cluster'].value_counts().sort_index())
    
    # Check label coverage in each cluster
    print(f"\nLabel Coverage in Training Clusters:")
    for cluster_id in range(n_clusters):
        if cluster_id in train_results:
            cluster_data = train_df_final[train_df_final['cluster'] == cluster_id]
            n_labels_in_cluster = len(cluster_data[target_column].unique())
            label_counts = cluster_data[target_column].value_counts()
            
            # Count labels below threshold
            labels_below = [(label, count) for label, count in label_counts.items() 
                           if count < min_threshold]
            
            print(f"  Cluster {cluster_id}:")
            print(f"    Labels covered: {n_labels_in_cluster}/{n_classes}")
            if labels_below:
                print(f"    Labels below threshold ({min_threshold}): {labels_below}")
    
    if test_results:
        print(f"\nOverall Test Performance (Weighted Average):")
        
        total_test_samples = sum(result['n_samples'] for result in test_results.values())
        
        metrics = ['accuracy', 'f1_weighted', 'recall_weighted', 'precision_weighted']
        overall_results = {}
        
        for metric in metrics:
            weighted_sum = sum(result[metric] * result['n_samples'] 
                             for result in test_results.values())
            overall_results[metric] = weighted_sum / total_test_samples
        
        print(f"  Total test samples: {total_test_samples}")
        print(f"  Accuracy: {overall_results['accuracy']:.4f}")
        print(f"  F1 Weighted: {overall_results['f1_weighted']:.4f}")
        print(f"  Recall Weighted: {overall_results['recall_weighted']:.4f}")
        print(f"  Precision Weighted: {overall_results['precision_weighted']:.4f}")
    
    print(f"\nTop Features per Cluster (based on original features):")
    for i, fi_df in feature_importances.items():
        if i in train_results:
            print(f"\n  Cluster {i}:")
            top_features_cluster = fi_df.head(3)
            for _, row in top_features_cluster.iterrows():
                print(f"    {row['feature']}: {row['importance']:.4f}")
    
    print("\n" + "="*80)
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print("="*80)
    
    # Return all results
    return {
        'train_df': train_df_final,
        'test_df': test_df_final,
        'classifiers': classifiers,
        'train_results': train_results,
        'test_results': test_results,
        'feature_importances': feature_importances,
        'top_features': top_features,  # Selected top features
        'all_original_features': all_features,
        'cluster_info': cluster_info,
        'n_classes': n_classes,
        'kmeans_model': kmeans,
        'scaler': scaler,
        'incremental_feature_cols': incremental_feature_cols,
        'trend_score_cols': trend_scores_train.columns.tolist(),
        'target_column': target_column,
        'preserve_order': preserve_order,
        'ensure_label_coverage': ensure_label_coverage,
        'min_threshold': min_threshold,
        'target_count': target_count,
        'top_n_features': top_n_features,
        'window_size': window_size  # FIXED: Added window_size to results
    }

# ------------------------------------------------------------
# 8. FIXED: DEPLOYMENT PREDICTION PIPELINE WITH WINDOW SIZE
# ------------------------------------------------------------

def create_deployment_pipeline(training_results):
    """
    FIXED: Create a deployment pipeline from training results
    Uses the SAME window_size from training
    """
    # Extract components from training results
    kmeans_model = training_results['kmeans_model']
    scaler = training_results['scaler']
    classifiers = training_results['classifiers']
    top_features = training_results['top_features']
    all_original_features = training_results['all_original_features']
    
    # FIXED: Get window_size from training results (now it's stored!)
    window_size = training_results.get('window_size')
    if window_size is None:
        raise ValueError("window_size not found in training results! Training must include window_size.")
    
    print("\n" + "="*60)
    print("DEPLOYMENT PIPELINE CREATED")
    print("="*60)
    print(f"Using top {len(top_features)} features selected during training:")
    for i, feat in enumerate(top_features):
        print(f"  {i+1}. {feat}")
    print(f"Each will generate 3 statistical features: {len(top_features) * 3} total")
    print(f"Using window size: {window_size} (SAME as training configuration)")
    
    # Initialize fresh states for deployment (starting from zero)
    # States for raw values (to calculate trend scores) - only for top features
    raw_states = {feature: [] for feature in top_features}
    
    # States for trend score statistics - one per top feature
    trend_states = []
    for _ in range(len(top_features)):
        trend_states.append({
            'n': 0,
            'mean': 0.0,
            'M2': 0.0,
            'median_calculator': ExpandingMedian()
        })
    
    def predict_new_sample(sample_dict):
        """
        FIXED: Predict label for a new sample with consistent window_size
        """
        # 1. Update raw values and calculate current trend scores
        trend_scores = []
        
        for f_idx, feature in enumerate(top_features):
            if feature in sample_dict:
                x = sample_dict[feature]
            else:
                raise ValueError(f"Selected top feature '{feature}' not found in sample")
            
            # Update raw values for this feature
            raw_states[feature].append(x)
            
            # FIXED: Proper window handling for trend score calculation
            if len(raw_states[feature]) < 2:
                # Not enough data for trend calculation
                score = 0.0
            elif len(raw_states[feature]) < window_size:
                # Partial window - use all available data
                score = calculate_trend_score(np.array(raw_states[feature]))
            else:
                # Full window - use last window_size values
                current_window = raw_states[feature][-window_size:]
                score = calculate_trend_score(np.array(current_window))
            
            # 2. Update trend score statistics
            state = trend_states[f_idx]
            state['n'] += 1
            n = state['n']
            
            # Update expanding mean
            delta = score - state['mean']
            state['mean'] += delta / n
            
            # Update expanding variance
            delta2 = score - state['mean']
            state['M2'] += delta * delta2
            
            # Update expanding median
            state['median_calculator'].add(score)
            median_val = state['median_calculator'].get_median()
            
            # Calculate expanding std
            if n > 1:
                variance = state['M2'] / (n - 1)
                std_val = np.sqrt(variance)
            else:
                std_val = 0.0
            
            # Add trend score statistics to feature vector (mean, median, std)
            trend_scores.extend([state['mean'], median_val, std_val])
        
        # 3. Scale features (using the same scaler from training)
        scaled_features = scaler.transform([trend_scores])
        
        # 4. Predict cluster (simple prediction, no edge priority)
        cluster_id = kmeans_model.predict(scaled_features)[0]
        
        # 5. Extract ALL original features for classification
        original_features = []
        for feature in all_original_features:
            if feature in sample_dict:
                original_features.append(sample_dict[feature])
            else:
                # If feature is missing, use 0 as default
                original_features.append(0.0)
        
        # 6. Predict using the classifier for this cluster
        result = {
            'cluster_id': int(cluster_id),
            'prediction': None,
            'confidence': None,
            'probabilities': None
        }
        
        if cluster_id in classifiers:
            classifier = classifiers[cluster_id]
            
            # Predict label
            prediction = classifier.predict([original_features])[0]
            result['prediction'] = int(prediction) if hasattr(prediction, '__int__') else prediction
            
            # Get probabilities if available
            if hasattr(classifier, 'predict_proba'):
                probabilities = classifier.predict_proba([original_features])[0]
                result['probabilities'] = probabilities.tolist()
                result['confidence'] = float(np.max(probabilities))
        
        return result
    
    return predict_new_sample

# ------------------------------------------------------------
# 9. VERIFICATION FUNCTION
# ------------------------------------------------------------

def verify_order_preservation(original_df, results, target_column='label'):
    """
    Verify that order is preserved in the pipeline for each class separately
    """
    train_df = results['train_df']
    test_df = results['test_df']
    
    print("\n" + "="*60)
    print("ORDER PRESERVATION VERIFICATION")
    print("="*60)
    
    # Get unique labels
    unique_labels = original_df[target_column].unique()
    
    all_correct = True
    
    for label in unique_labels:
        print(f"\nClass {label}:")
        
        # Get original indices for this class
        original_class_indices = original_df[original_df[target_column] == label].index
        
        # Get result indices for this class
        train_class_indices = train_df[train_df[target_column] == label].index
        test_class_indices = test_df[test_df[target_column] == label].index
        
        # Combine result indices
        result_class_indices = list(train_class_indices) + list(test_class_indices)
        
        # Check 1: All indices are preserved
        if set(original_class_indices) == set(result_class_indices):
            print(f"  ✓ All indices preserved")
        else:
            print(f"  ✗ Some indices missing!")
            print(f"    Missing: {set(original_class_indices) - set(result_class_indices)}")
            all_correct = False
        
        # Check 2: Indices are in order (train before test)
        if len(train_class_indices) > 0 and len(test_class_indices) > 0:
            max_train_idx = max(train_class_indices)
            min_test_idx = min(test_class_indices)
            
            if max_train_idx < min_test_idx:
                print(f"  ✓ Train indices come before test indices")
                print(f"    Train: {len(train_class_indices)} samples (indices up to {max_train_idx})")
                print(f"    Test: {len(test_class_indices)} samples (indices from {min_test_idx})")
            else:
                print(f"  ✗ Order violation: Test indices appear before train indices")
                print(f"    Max train index: {max_train_idx}")
                print(f"    Min test index: {min_test_idx}")
                all_correct = False
        else:
            print(f"  ✓ Only one set has data (train: {len(train_class_indices)}, test: {len(test_class_indices)})")
    
    print(f"\n" + "="*60)
    if all_correct:
        print("✓ ORDER PRESERVATION VERIFIED FOR ALL CLASSES")
    else:
        print("⚠ WARNING: ORDER PRESERVATION ISSUES DETECTED")
    print("="*60)
    
    return all_correct

# ------------------------------------------------------------
# 10. EXECUTION FUNCTIONS
# ------------------------------------------------------------

def run_mode_1_single_file(file_path, preserve_order=True, **kwargs):
    """
    MODE 1: Read a single CSV file and split into train/test internally
    ALWAYS preserves order by default for trend score calculation
    """
    print("="*80)
    print("MODE 1: SINGLE FILE TRAIN/TEST SPLIT")
    print("="*80)
    print(f"Note: Order preservation = {preserve_order}")
    if preserve_order:
        print("      Using CLASS-WISE sequential split")
    
    df = pd.read_csv(file_path)
    print(f"✓ Loaded data from: {file_path}")
    print(f"✓ Data shape: {df.shape}")
    print(f"✓ Columns: {df.columns.tolist()}")
    
    # Ensure preserve_order is passed to pipeline
    kwargs['preserve_order'] = preserve_order
    
    results = universal_trend_clustering_pipeline(df=df, **kwargs)
    return results

def run_mode_2_two_files(train_file_path, test_file_path, **kwargs):
    """
    MODE 2: Read two separate CSV files for train and test
    Order is preserved from the original files
    """
    print("="*80)
    print("MODE 2: SEPARATE TRAIN/TEST FILES")
    print("="*80)
    print("Note: Order is preserved from original files")
    
    # Load data
    train_df = pd.read_csv(train_file_path)
    test_df = pd.read_csv(test_file_path)
    
    print(f"✓ Train file: {train_file_path}, Shape: {train_df.shape}")
    print(f"✓ Test file: {test_file_path}, Shape: {test_df.shape}")
    
    # Ensure both have the same columns
    train_cols = set(train_df.columns)
    test_cols = set(test_df.columns)
    
    if train_cols != test_cols:
        print("⚠ Warning: Train and test files have different columns!")
        print(f"  Train columns: {train_df.columns.tolist()}")
        print(f"  Test columns: {test_df.columns.tolist()}")
        
        # Keep only common columns
        common_cols = list(train_cols.intersection(test_cols))
        print(f"✓ Using common columns: {common_cols}")
        
        if len(common_cols) < 2:
            raise ValueError("✗ Not enough common columns between train and test files!")
        
        train_df = train_df[common_cols]
        test_df = test_df[common_cols]
    
    # Run pipeline with external test set
    results = universal_trend_clustering_pipeline(
        df=train_df,
        external_test_df=test_df,
        preserve_order=True,
        **kwargs
    )
    return results

def run_mode_3_dataframes(train_df, test_df=None, preserve_order=True, **kwargs):
    """
    MODE 3: Use existing DataFrames
    """
    if test_df is None:
        print("="*80)
        print("MODE 3: SINGLE DATAFRAME")
        print("="*80)
        print(f"Note: Order preservation = {preserve_order}")
        if preserve_order:
            print("      Using CLASS-WISE sequential split")
        
        kwargs['preserve_order'] = preserve_order
        results = universal_trend_clustering_pipeline(df=train_df, **kwargs)
    else:
        print("="*80)
        print("MODE 3: TWO DATAFRAMES")
        print("="*80)
        print("Note: Order is preserved from input DataFrames")
        
        results = universal_trend_clustering_pipeline(
            df=train_df,
            external_test_df=test_df,
            preserve_order=True,
            **kwargs
        )
    return results

def create_demo_data(n_samples=1000, n_features=10, n_classes=2):
    """
    Create demo data for testing
    """
    np.random.seed(42)
    
    # Create synthetic data with trends
    demo_data = {}
    for i in range(n_features):
        # Create data with different trends
        if i % 3 == 0:
            # Increasing trend
            base = np.linspace(0, 10, n_samples)
            noise = np.random.randn(n_samples) * 0.5
            demo_data[f'feature_{i}'] = base + noise
        elif i % 3 == 1:
            # Decreasing trend
            base = np.linspace(10, 0, n_samples)
            noise = np.random.randn(n_samples) * 0.5
            demo_data[f'feature_{i}'] = base + noise
        else:
            # No clear trend
            demo_data[f'feature_{i}'] = np.random.randn(n_samples)
    
    # Create labels
    if n_classes == 2:
        demo_data['label'] = np.random.choice([0, 1], size=n_samples, p=[0.7, 0.3])
    elif n_classes == 3:
        demo_data['label'] = np.random.choice([0, 1, 2], size=n_samples, p=[0.6, 0.3, 0.1])
    else:
        demo_data['label'] = np.random.choice(range(n_classes), size=n_samples)
    
    demo_df = pd.DataFrame(demo_data)
    return demo_df

# ------------------------------------------------------------
# 11. EXAMPLE USAGE (FIXED)
# ------------------------------------------------------------

if __name__ == "__main__":
    # Create demo data
    print("Creating demo data...")
    demo_df = create_demo_data(n_samples=500, n_features=8, n_classes=3)
    
    # Run pipeline with top 3 features
    results = universal_trend_clustering_pipeline(
        df=demo_df,
        target_column='label',
        n_clusters=3,
        top_n_features=3,
        window_size=5,  # FIXED: window_size now works correctly
        test_size=0.2,
        preserve_order=True,
        ensure_label_coverage=True,
        min_threshold=10,
        target_count=50
    )
    
    # Verify order preservation
    verify_order_preservation(demo_df, results, target_column='label')
    
    # Create deployment pipeline
    predict_func = create_deployment_pipeline(results)
    
    # Test prediction on a new sample
    print("\n" + "="*60)
    print("TESTING DEPLOYMENT PIPELINE")
    print("="*60)
    
    # Get a sample from test set
    test_sample = results['test_df'].iloc[0].to_dict()
    print(f"Sample features: {list(test_sample.keys())}")
    
    # Make prediction
    prediction_result = predict_func(test_sample)
    print(f"\nPrediction result:")
    print(f"  Cluster ID: {prediction_result['cluster_id']}")
    print(f"  Predicted label: {prediction_result['prediction']}")
    print(f"  Confidence: {prediction_result['confidence']:.4f}")
    
    # Your code with two DataFrames
    print("\n" + "="*80)
    print("RUNNING WITH YOUR DATAFRAMES")
    print("="*80)


Creating demo data...
UNIVERSAL TREND CLUSTERING & CLASSIFICATION PIPELINE
USING TOP 3 FEATURES WITH HIGHEST TREND STRENGTH
Each selected feature → 3 statistical features (mean, median, std)
Total clustering features: 3 × 3 = 9
Trend window size: 5 (used consistently throughout)
NOTE: Data order preservation = True
NOTE: Label coverage enforcement = True
      Enhancement threshold: <10 samples
      Target count per label: 50 samples

[1] DATA PREPARATION
    Target column: label
    Number of features: 8
    Total samples: 500
    Number of classes: 3
    Class distribution:
      Class 0: 313 samples (62.6%)
      Class 1: 134 samples (26.8%)
      Class 2: 53 samples (10.6%)

[2] DATA SPLITTING
    Using CLASS-WISE sequential split (order preserved)
    Splitting each of 3 classes separately...
      Class 0: 313 total -> 250 train, 63 test
      Class 2: 53 total -> 42 train, 11 test
      Class 1: 134 total -> 107 train, 27 test

    Training samples: 399
    Test samples: 101

 

In [7]:
# ------------------------------------------------------------
# MAIN EXECUTION WITH COMMAND LINE INTERFACE SUPPORT
# ------------------------------------------------------------

    
# Example usage with your data
df = pd.read_csv('DATASETS/ISCX-TOR/ISCX-TOR.csv')

# Quick usage with your own DataFrame
# Example 1: Single DataFrame with internal split
results = run_mode_3_dataframes(
    train_df=df,
    target_column='label',
    n_clusters=3,
    top_n_features=3,
    test_size=0.2,
    window_size = 1,
)

# Example 2: Two DataFrames (train and test already separated)
# train_data = pd.read_csv('train.csv')
# test_data = pd.read_csv('test.csv')
# results = run_mode_3_dataframes(
#     train_df=train_data,
#     test_df=test_data,
#     target_column='label',
#     n_clusters=4,
#     top_n_features=5
# )

MODE 3: SINGLE DATAFRAME
Note: Order preservation = True
      Using CLASS-WISE sequential split
UNIVERSAL TREND CLUSTERING & CLASSIFICATION PIPELINE
USING TOP 3 FEATURES WITH HIGHEST TREND STRENGTH
Each selected feature → 3 statistical features (mean, median, std)
Total clustering features: 3 × 3 = 9
Trend window size: 1 (used consistently throughout)
NOTE: Data order preservation = True
NOTE: Label coverage enforcement = True
      Enhancement threshold: <10 samples
      Target count per label: 1 samples

[1] DATA PREPARATION
    Target column: label
    Number of features: 36
    Total samples: 14414
    Number of classes: 8
    Class distribution:
      Class audio: 1035 samples (7.2%)
      Class browsing: 1887 samples (13.1%)
      Class chat: 502 samples (3.5%)
      Class ftp: 1665 samples (11.6%)
      Class mail: 461 samples (3.2%)
      Class p2p: 2138 samples (14.8%)
      Class video: 2202 samples (15.3%)
      Class voip: 4524 samples (31.4%)

[2] DATA SPLITTING
    Usin

In [15]:
# ------------------------------------------------------------
# MAIN EXECUTION WITH COMMAND LINE INTERFACE SUPPORT
# ------------------------------------------------------------

    
# Example usage with your data
#df = pd.read_csv('DATASETS/CTU-13/5_virut.csv')

# Quick usage with your own DataFrame
# Example 1: Single DataFrame with internal split
#results = run_mode_3_dataframes(
#    train_df=df,
#    target_column='your_target_column',
#    n_clusters=3,
#    top_n_features=3,
#    test_size=0.2
#)

# Example 2: Two DataFrames (train and test already separated)
train_data = pd.read_csv('DATASETS/CDR-MLC/level_1.csv')
test_data = pd.read_csv('DATASETS/CDR-MLC/level_3.csv')
results = run_mode_3_dataframes(
    train_df=train_data,
    test_df=test_data,
    target_column='label',
    n_clusters=1,
    top_n_features=1,
    window_size=1,
)

MODE 3: TWO DATAFRAMES
Note: Order is preserved from input DataFrames
UNIVERSAL TREND CLUSTERING & CLASSIFICATION PIPELINE
USING TOP 1 FEATURES WITH HIGHEST TREND STRENGTH
Each selected feature → 3 statistical features (mean, median, std)
Total clustering features: 1 × 3 = 3
Trend window size: 1 (used consistently throughout)
NOTE: Data order preservation = True
NOTE: Label coverage enforcement = True
      Enhancement threshold: <10 samples
      Target count per label: 1 samples

[1] DATA PREPARATION
    Target column: label
    Number of features: 36
    Total samples: 68079
    Number of classes: 5
    Class distribution:
      Class HTTP: 39798 samples (58.5%)
      Class SFTP: 11422 samples (16.8%)
      Class SMTP: 6267 samples (9.2%)
      Class SSH: 3127 samples (4.6%)
      Class VIDEO: 7465 samples (11.0%)

[2] DATA SPLITTING
    Using external test set provided
    Training samples: 68079
    Test samples: 60482

[3] FEATURE SELECTION BASED ON TREND STRENGTH
    Analyzing a